# Lecture 09: Weinstein Tubular Neighborhood Theorem

**Source span.** Printed pages 49-54; physical PDF pages 63-68 in `Lectures on Symplectic Geometry.pdf`. I inspected the local PDF text for this span before revising the notebook.

**Lecture goal.** Show how a Lagrangian submanifold has a canonical cotangent-neighborhood model, then use that model to read small symplectomorphisms as closed one-forms and fixed points as graph-diagonal intersections.

The lecture starts with a linear algebra observation: if `U` is Lagrangian in `(V, Omega)`, then the quotient `V/U` pairs nondegenerately with `U`, so the normal bundle `N X` of a Lagrangian submanifold is canonically identified with `T^*X`. That is the bridge from ordinary tubular neighborhoods to the Weinstein tubular neighborhood theorem. The second half applies the same bridge to the diagonal in `M x M`: a graph near the diagonal becomes a one-form near the zero section, and the graph is Lagrangian exactly when that one-form is closed.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, read_json, save_json, save_matplotlib
from utils.symplectic import standard_omega

ARTIFACT_TOPIC = "lecture-09"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact topic: {ARTIFACT_TOPIC}")

## Translation Guide And Library Routing

| Source idea | Computational object | Check |
| --- | --- | --- |
| symplectic normal bundle | quotient representatives modulo a Lagrangian subspace | adding a vector in `U` does not change the pairing |
| `N X ~= T^*X` | pairing matrix from normal directions to covectors on `U` | pairing matrix is invertible |
| graph near diagonal | graph basis in the product form `pr1^* omega - pr2^* omega` | `Graph(F)` is Lagrangian iff `F.T @ J @ F = J` |
| closed one-form tangent model | vector field section over the base | `d mu = 0`, implemented as curl zero in a local chart |
| fixed-point intersection | zeros of the one-form / critical points of a potential | graph meets the diagonal exactly where `mu = 0` |
| Arnold correspondence | time-one Hamiltonian map and period-one orbits | fixed point of `f` is a closed orbit of the isotopy |

**Library routing.** `numpy` handles the quotient-pairing, product-graph, and curl residuals. `matplotlib` is used for the normal/cotangent pairing, graph-near-diagonal model, and fixed-point detector because each visual is a static geometric inspection target. `networkx` appears once for the theorem-dependency and Arnold-conjecture ledger.

## Visualization Storyboard

1. **Normal-to-cotangent pairing.** A finite model of the linear algebra observation checks that `V/U` pairs nondegenerately with `U` and that the pairing is well-defined modulo `U`.
2. **Graph near diagonal and closed one-form tangent model.** A product-form residual distinguishes symplectic graphs from non-symplectic graphs, then a one-form panel shows why closedness is the local tangent condition.
3. **Fixed-point intersection and Arnold ledger.** A potential function gives an exact one-form; zeros of `dh` are the fixed-point candidates. The proof graph records the path from Weinstein's neighborhood model to closed one-forms, exact forms, critical points, and the Arnold/Floer bound language.
4. **Classification remarks.** Isotropic and coisotropic embedding classifications are summarized as normal-form outputs rather than expanded into copied theorem text.

In [ ]:
# Linear algebra observation: V/U pairs nondegenerately with U.
J = standard_omega(2)
U = np.eye(4)[:, [0, 1]]      # Lagrangian q-plane, columns q1 and q2
normal_reps = np.eye(4)[:, [2, 3]]  # quotient representatives p1 and p2
pairing = normal_reps.T @ J @ U
pairing_det = float(np.linalg.det(pairing))
well_defined_residual = float(np.linalg.norm((normal_reps + U @ np.array([[0.7, -0.2], [0.1, 0.4]])).T @ J @ U - pairing))
assert abs(pairing_det) > 0.99
assert well_defined_residual < 1e-12

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
im = axes[0].imshow(pairing, cmap="BrBG", vmin=-1.2, vmax=1.2)
axes[0].set_title("pairing V/U x U -> R")
axes[0].set_xticks([0, 1], labels=["q1", "q2"])
axes[0].set_yticks([0, 1], labels=["[p1]", "[p2]"])
for (i, j), value in np.ndenumerate(pairing):
    axes[0].text(j, i, f"{value:.0f}", ha="center", va="center", fontsize=13)
fig.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].quiver([0, 0], [0, 0], [1, 0], [0, 1], color="#1b9e77", angles="xy", scale_units="xy", scale=1, label="U directions")
axes[1].quiver([0, 0], [0, 0], [0, -1], [1, 0], color="#d95f02", angles="xy", scale_units="xy", scale=1, label="dual normal covectors")
axes[1].set_xlim(-1.3, 1.3)
axes[1].set_ylim(-1.3, 1.3)
axes[1].set_aspect("equal", adjustable="box")
axes[1].axhline(0, color="0.82", lw=0.8)
axes[1].axvline(0, color="0.82", lw=0.8)
axes[1].set_title("normal directions become covectors on X")
axes[1].legend(loc="upper right", fontsize=8)
fig.suptitle("Theorem 9.1: the symplectic normal bundle is canonically T*X")
normal_pairing_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "normal-cotangent-pairing.png")
plt.close(fig)
display_artifact(normal_pairing_path, width=760)
print({"pairing_det": pairing_det, "well_defined_residual": well_defined_residual})

In [ ]:
# Graph near the diagonal: product Lagrangian test and one-form closedness.
def product_graph_residual(F):
    basis = np.vstack([np.eye(2), F])
    J2 = standard_omega(1)
    product_form = np.block([[J2, np.zeros((2, 2))], [np.zeros((2, 2)), -J2]])
    return float(np.linalg.norm(basis.T @ product_form @ basis))

angle = 0.22
F_sympl = np.array([[np.cos(angle), np.sin(angle)], [-np.sin(angle), np.cos(angle)]])
F_bad = np.array([[1.0, 0.35], [0.05, 1.0]])
graph_residual_sympl = product_graph_residual(F_sympl)
graph_residual_bad = product_graph_residual(F_bad)
assert graph_residual_sympl < 1e-12
assert graph_residual_bad > 1e-2

x = np.linspace(-1.2, 1.2, 21)
y = np.linspace(-1.2, 1.2, 21)
X, Y = np.meshgrid(x, y)
# exact one-form dh for h = 0.5*x^2 - 0.35*y^2 + 0.15*x*y
mu_exact_x = X + 0.15 * Y
mu_exact_y = -0.70 * Y + 0.15 * X
curl_exact = 0.15 - 0.15
mu_bad_x = -Y
mu_bad_y = X
curl_bad = 1.0 - (-1.0)
assert abs(curl_exact) < 1e-12
assert abs(curl_bad - 2.0) < 1e-12

pts = np.array([[np.cos(t), np.sin(t)] for t in np.linspace(0, 2*np.pi, 80)])
image_sympl = pts @ F_sympl.T
image_bad = pts @ F_bad.T

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
axes[0].plot(pts[:, 0], pts[:, 1], color="#2c7fb8", lw=2, label="near diagonal")
axes[0].plot(image_sympl[:, 0], image_sympl[:, 1], color="#1b9e77", lw=2, label="symplectic graph")
axes[0].plot(image_bad[:, 0], image_bad[:, 1], color="#d95f02", lw=2, label="non-symplectic graph")
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title("Graph(F) Lagrangian test")
axes[0].legend(fontsize=8)

axes[1].quiver(X, Y, mu_exact_x, mu_exact_y, color="#1b9e77", angles="xy")
axes[1].set_title("closed one-form model: d mu = 0")
axes[1].set_aspect("equal", adjustable="box")

axes[2].quiver(X, Y, mu_bad_x, mu_bad_y, color="#d95f02", angles="xy")
axes[2].set_title("rotating one-form: d mu != 0")
axes[2].set_aspect("equal", adjustable="box")
for ax in axes:
    ax.axhline(0, color="0.85", lw=0.8)
    ax.axvline(0, color="0.85", lw=0.8)
    ax.set_xlabel("local coordinate 1")
    ax.set_ylabel("local coordinate 2")
fig.suptitle("A C1-neighborhood of id in Sympl(M,omega) is modeled by closed one-forms")
graph_one_form_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "graph-near-diagonal-closed-one-form-model.png")
plt.close(fig)
display_artifact(graph_one_form_path, width=760)
print({"graph_residual_sympl": graph_residual_sympl, "graph_residual_bad": graph_residual_bad, "curl_bad": curl_bad})

In [ ]:
# Fixed-point detector and Arnold-conjecture proof ledger.
xx = np.linspace(-np.pi, np.pi, 220)
yy = np.linspace(-np.pi, np.pi, 220)
XX, YY = np.meshgrid(xx, yy)
H = np.cos(XX) + 0.65 * np.cos(YY)
dHx = -np.sin(XX)
dHy = -0.65 * np.sin(YY)
critical_points = [(a, b) for a in [0, np.pi, -np.pi] for b in [0, np.pi, -np.pi]]
unique_critical_points = []
for a, b in critical_points:
    representative = (0.0 if abs(a) < 1e-12 else np.pi, 0.0 if abs(b) < 1e-12 else np.pi)
    if representative not in unique_critical_points:
        unique_critical_points.append(representative)
critical_count_model = len(unique_critical_points)
assert critical_count_model >= 2

G = nx.DiGraph()
G.add_edges_from([
    ("Weinstein tubular neighborhood", "graph near diagonal becomes section of T*M"),
    ("graph near diagonal becomes section of T*M", "small symplectomorphism -> one-form mu"),
    ("Graph f is Lagrangian", "d mu = 0"),
    ("d mu = 0", "T_id Sympl(M,omega) = closed one-forms"),
    ("H1(M)=0", "mu = dh"),
    ("mu = dh", "fixed points = critical points of h"),
    ("compact M", "at least two critical points"),
    ("fixed points = critical points of h", "near-identity fixed point theorem"),
    ("Hamiltonian isotopy", "time-one map f"),
    ("time-one map f", "fixed points = period-one orbits"),
    ("Morse/Floer theory", "Arnold lower bounds"),
    ("fixed points = period-one orbits", "Arnold lower bounds"),
])
pos = {
    "Weinstein tubular neighborhood": (0, 2.4),
    "graph near diagonal becomes section of T*M": (2.0, 2.4),
    "small symplectomorphism -> one-form mu": (4.0, 2.4),
    "Graph f is Lagrangian": (2.0, 1.3),
    "d mu = 0": (4.0, 1.3),
    "T_id Sympl(M,omega) = closed one-forms": (6.2, 1.3),
    "H1(M)=0": (2.0, 0.1),
    "mu = dh": (4.0, 0.1),
    "fixed points = critical points of h": (6.2, 0.1),
    "compact M": (4.0, -1.1),
    "at least two critical points": (6.2, -1.1),
    "near-identity fixed point theorem": (8.3, -0.45),
    "Hamiltonian isotopy": (0, -2.2),
    "time-one map f": (2.0, -2.2),
    "fixed points = period-one orbits": (4.0, -2.2),
    "Morse/Floer theory": (6.2, -2.2),
    "Arnold lower bounds": (8.3, -2.2),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6.4), gridspec_kw={"width_ratios": [1, 1.45]})
contours = axes[0].contour(XX, YY, H, levels=18, cmap="viridis")
axes[0].clabel(contours, inline=True, fontsize=7)
for a, b in unique_critical_points:
    axes[0].plot(a, b, "o", color="#d95f02", markersize=7)
axes[0].set_title("fixed-point candidates as zeros of dh")
axes[0].set_xlabel("local coordinate q1")
axes[0].set_ylabel("local coordinate q2")
axes[0].set_aspect("equal", adjustable="box")

nx.draw_networkx_edges(G, pos, ax=axes[1], arrows=True, arrowstyle="-|>", arrowsize=13, edge_color="0.38")
nx.draw_networkx_nodes(G, pos, ax=axes[1], node_color="#c7e9c0", node_size=1700, edgecolors="0.25", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=axes[1], font_size=7)
axes[1].set_axis_off()
axes[1].set_title("from graph intersections to Arnold's fixed-point viewpoint")
fig.suptitle("Fixed-point intersection: Graph(f) meets the diagonal where the associated one-form vanishes")
fixed_point_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "fixed-point-intersection-arnold-ledger.png")
plt.close(fig)
display_artifact(fixed_point_path, width=760)
print({"critical_count_model": critical_count_model})

## Classification And Application Notes

The source span also records neighboring classification results. Lagrangian embeddings are classified locally by their zero-section cotangent model. Isotropic embeddings require symplectic vector-bundle data. Coisotropic embeddings use a closed two-form of constant rank and its characteristic distribution; the model lives on the dual bundle to that distribution near the zero section. The notebook keeps these as a compact normal-form map because the full classification proofs belong to later or external references.

For symplectomorphisms, the key local replacement is `Graph f` near the diagonal. In `M x M` with product form `pr1^* omega - pr2^* omega`, both the diagonal and the graph of a symplectomorphism are Lagrangian. Weinstein's theorem turns a graph close to the diagonal into a one-form `mu`; the Lagrangian condition becomes `d mu = 0`. If `H^1_deRham(M)=0`, closed means exact, so `mu = dh`, and fixed points become critical points of `h`. This is the near-identity fixed-point theorem that foreshadows Arnold's conjecture and Floer homology.

In [ ]:
residuals = {
    "normal_to_cotangent_pairing_det": pairing_det,
    "pairing_well_defined_residual": well_defined_residual,
    "graph_residual_symplectic_linear_map": graph_residual_sympl,
    "graph_residual_non_symplectic_linear_map": graph_residual_bad,
    "closed_one_form_curl": curl_exact,
    "nonclosed_one_form_curl": curl_bad,
    "model_critical_point_count": critical_count_model,
}
residual_path = save_json(residuals, ARTIFACT_TOPIC, "checks", "weinstein-tubular-residuals.json")

source_span = {
    "title": "Weinstein Tubular Neighborhood Theorem",
    "printed_pages": "49-54",
    "physical_pdf_pages": "63-68",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "sections": [
        "Observation from Linear Algebra",
        "Tubular Neighborhoods",
        "Tangent Space to the Group of Symplectomorphisms",
        "Fixed Points of Symplectomorphisms",
        "Arnold conjecture overview",
    ],
    "concepts": [
        "symplectic normal bundle",
        "graph near diagonal",
        "closed one-form tangent model",
        "fixed-point intersection",
        "Arnold fixed-point correspondence",
    ],
    "copyright_note": "Source pages were used for terminology and coverage; notebook prose, visuals, and code are original and no page crops or long exercise text are copied.",
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Turn Weinstein tubular neighborhoods into testable normal/cotangent, graph, one-form, and fixed-point models.",
    "source_span_read": source_span,
    "library_routing": [
        {"concept": "symplectic normal bundle", "representation": "quotient pairing heatmap", "library": "numpy + matplotlib", "why": "the theorem begins with finite-dimensional pairing algebra"},
        {"concept": "graph near diagonal", "representation": "product-form graph residual", "library": "numpy + matplotlib", "why": "the Lagrangian graph criterion is a direct matrix residual"},
        {"concept": "closed one-form tangent model", "representation": "one-form quiver panels and curl checks", "library": "numpy + matplotlib", "why": "closedness is visible as zero local curl"},
        {"concept": "fixed-point intersection", "representation": "potential contours plus proof dependency graph", "library": "matplotlib + networkx", "why": "zeros of dh and theorem dependencies must be inspected together"},
    ],
    "visual_sequence": [
        {"concept": "symplectic normal bundle", "artifact": "artifacts/lecture-09/figures/normal-cotangent-pairing.png", "inspection_target": "normal representatives pair nondegenerately with tangent directions"},
        {"concept": "graph near diagonal and closed one-form tangent model", "artifact": "artifacts/lecture-09/figures/graph-near-diagonal-closed-one-form-model.png", "inspection_target": "symplectic graphs correspond to closed one-form sections"},
        {"concept": "fixed-point intersection", "artifact": "artifacts/lecture-09/figures/fixed-point-intersection-arnold-ledger.png", "inspection_target": "fixed points are zeros of the associated exact one-form and critical points of a potential"},
    ],
    "computational_checks": residuals,
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")

final_sanity = {
    "artifacts": [
        str(normal_pairing_path.relative_to(BOOK_ROOT)),
        str(graph_one_form_path.relative_to(BOOK_ROOT)),
        str(fixed_point_path.relative_to(BOOK_ROOT)),
        str(residual_path.relative_to(BOOK_ROOT)),
        str(source_path.relative_to(BOOK_ROOT)),
        str(storyboard_path.relative_to(BOOK_ROOT)),
    ],
    "assertions": {
        "normal_pairing_is_nondegenerate": abs(pairing_det) > 0.99,
        "pairing_is_well_defined_mod_U": well_defined_residual < 1e-12,
        "symplectic_graph_is_lagrangian": graph_residual_sympl < 1e-12,
        "non_symplectic_graph_fails_lagrangian_test": graph_residual_bad > 1e-2,
        "closed_one_form_has_zero_curl": abs(curl_exact) < 1e-12,
        "nonclosed_one_form_has_nonzero_curl": abs(curl_bad) > 1.0,
        "fixed_point_model_has_at_least_two_zeros": critical_count_model >= 2,
    },
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print(json.dumps(final_sanity, indent=2))

In [ ]:
final_sanity = read_json(ARTIFACT_ROOT / "checks" / "final-sanity.json")
assert all(final_sanity["assertions"].values())
for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    assert path.exists(), relative
    assert path.stat().st_size > 100, relative

for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    if path.suffix.lower() in {".png", ".html", ".svg"}:
        display_artifact(path, width=760, height=430 if path.suffix == ".html" else None)

print("Lecture 09 checks passed; artifacts are present and nonempty.")

## Takeaways

- The canonical pairing `V/U x U -> R` is the linear reason a Lagrangian normal bundle is `T^*X`.
- The Weinstein tubular neighborhood theorem refines the ordinary tubular theorem by preserving the symplectic form, not just the underlying smooth neighborhood.
- Near the identity, symplectomorphisms are modeled by closed one-forms; exact one-forms are generated by functions modulo locally constant functions.
- Fixed points become intersections of the graph with the diagonal, or zeros of the corresponding one-form. When `H^1` vanishes, this becomes a critical-point problem.
- Arnold's conjecture extends this local near-identity picture to Hamiltonian time-one maps, where fixed points correspond to period-one Hamiltonian orbits and Morse/Floer theory supplies lower bounds.